# DANFront - Transylvania (First Military Survey): control-point crops

The Transylvania DVD has no georeferencing. The sheets form a regular grid, which was fitted with a few control points
(about 1.5 km off on average). To correct the grid piece by piece, more control points are needed.

This notebook downloads the ~40 sheets that contain a well-known town, cuts a small picture around the place where the town
should be, and packs the pictures into one zip file. Download the zip and give it to Claude. A normal CPU runtime is enough,
no R2 keys and no Drive needed. Takes about 15 minutes.

In [ ]:
!pip -q install rasterio pyproj
import os, sys
os.makedirs('/content/work', exist_ok=True); os.chdir('/content/work'); sys.path.insert(0, '/content/work')

In [ ]:
%%writefile /content/work/danfront_funcs.py
# ---- helper functions (pure Python; no Colab-specific calls) ----
import csv, json, math, os, shutil, subprocess, sys, time
import requests

def free_gb(path):
    return shutil.disk_usage(path).free / 1e9

def wait_for_disk(path, min_free=12, resume_at=20):
    """Pause while the local disk is nearly full (Drive needs time to sync its write cache)."""
    if free_gb(path) >= min_free:
        return
    print(f"  local disk low ({free_gb(path):.0f} GB free) - waiting for Drive to catch up ...")
    while free_gb(path) < resume_at:
        time.sleep(30)
    print("  continuing")

def download_parts(url, dest_prefix, part_size=4 * 1024**3, chunk=8 * 1024**2, disk_path="/content"):
    """Download url as raw volumes dest_prefix.001, .002, ... (7-Zip opens these directly).
    Finished parts are skipped, so the cell can be re-run with a fresh link after an interruption."""
    s = requests.Session()
    r = s.get(url, headers={"Range": "bytes=0-0"}, stream=True, timeout=60)
    cr = r.headers.get("Content-Range"); status = r.status_code; ctype = r.headers.get("Content-Type", ""); r.close()
    if status in (401, 403) or "text/html" in ctype:
        raise RuntimeError(f"The link was refused (HTTP {status}, {ctype}). It has probably expired: copy a fresh download link and run this cell again.")
    if status != 206 or not cr:
        raise RuntimeError(f"The server did not accept a ranged request (HTTP {status}). Send this message to Claude.")
    total = int(cr.split("/")[-1])
    size_file = dest_prefix + ".size"
    if os.path.exists(size_file):
        old = int(open(size_file).read().strip())
        if old != total:
            raise RuntimeError(f"This link points to a file of {total} bytes, but the parts already downloaded belong to a file of {old} bytes. Wrong link?")
    else:
        open(size_file, "w").write(str(total))
    nparts = math.ceil(total / part_size)
    print(f"archive size {total/1e9:.1f} GB -> {nparts} parts")
    t0 = time.time(); done_bytes = 0; new_bytes = 0
    for k in range(nparts):
        start = k * part_size; end = min(total, start + part_size) - 1; want = end - start + 1
        part = f"{dest_prefix}.{k+1:03d}"
        if os.path.exists(part) and os.path.getsize(part) == want:
            done_bytes += want; continue
        tmp = part + ".tmp"
        for attempt in range(5):
            try:
                r = s.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=120)
                if r.status_code != 206:
                    raise RuntimeError(f"HTTP {r.status_code}")
                got = 0
                with open(tmp, "wb") as f:
                    for c in r.iter_content(chunk):
                        f.write(c); got += len(c)
                        wait_for_disk(disk_path)
                if got != want:
                    raise IOError(f"got {got} of {want} bytes")
                os.replace(tmp, part); break
            except Exception as e:
                print(f"  part {k+1}: {e} - retrying ({attempt+1}/5)"); time.sleep(20)
        else:
            raise RuntimeError("Download keeps failing. The link has probably expired: copy a fresh download link and run this cell again (finished parts are kept).")
        done_bytes += want; new_bytes += want
        rate = new_bytes / max(1, time.time() - t0) / 1e6
        print(f"part {k+1}/{nparts} done  ({done_bytes/1e9:.1f} of {total/1e9:.1f} GB, {rate:.0f} MB/s)")
    have = sum(os.path.getsize(f"{dest_prefix}.{k+1:03d}") for k in range(nparts))
    assert have == total, f"size mismatch: {have} != {total}"
    print("download complete and verified:", f"{total/1e9:.1f} GB")

def list_archive(first_volume, exe="7z"):
    """Return (properties dict, [(path, size)] of GeoTIFFs with coordinates) from the archive listing."""
    out = subprocess.run([exe, "l", "-slt", first_volume], capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError("7z could not read the archive:\n" + out.stdout[-2000:] + out.stderr[-2000:])
    head, _, body = out.stdout.partition("----------")
    props = dict(l.split(" = ", 1) for l in head.splitlines() if " = " in l)
    sheets = []
    for block in body.split("\n\n"):
        e = dict(l.split(" = ", 1) for l in block.splitlines() if " = " in l)
        p = e.get("Path", "")
        if not p.lower().endswith((".tif", ".tiff")) or e.get("Folder") == "+" or e.get("Attributes", "").startswith("D"):
            continue
        if "/raw/" in "/" + p.replace("\\", "/"):
            continue                      # RAW sheets (no coordinates) are not for this notebook
        sheets.append((p, int(e.get("Size", 0))))
    return props, sorted(sheets)

def load_done(done_csv):
    if not os.path.exists(done_csv):
        return {}
    with open(done_csv, newline="", encoding="utf-8") as f:
        return {r["path"]: r for r in csv.DictReader(f)}

DONE_FIELDS = ["path", "sheet", "status", "cell_i", "cell_j", "cell_coverage", "tiles"]

def run_batches(cfg, max_sheets=None, batch_size=12, bbox=None, workers=2):
    """Extract -> tile -> upload -> record, batch by batch. Safe to interrupt and re-run."""
    import importlib
    tb = importlib.import_module(cfg.get("tiler", "tiles_2kf_batch"))   # tiles_1kf_batch for the First Survey
    done = load_done(cfg["done_csv"])
    todo = [p for p, _ in cfg["sheets"] if p not in done]
    if max_sheets:
        todo = todo[:max_sheets]
    print(f"{len(done)} sheets already done, {len(todo)} to do in this run")
    if (done or cfg.get("restore_always")) and not os.path.exists(cfg["restored_marker"]):
        print("new session: fetching the tiles made so far back from R2 (needed to merge sheet edges) ...")
        rc = subprocess.run(["rclone", "copy", cfg["remote"], cfg["tiles"], "--transfers", "64", "--checkers", "64"]).returncode
        if rc != 0:
            raise RuntimeError("could not restore tiles from R2")
    os.makedirs(cfg["tiles"], exist_ok=True); open(cfg["restored_marker"], "w").close()
    bounds = json.load(open(cfg["bounds_json"])) if os.path.exists(cfg["bounds_json"]) else {}
    bounds = {int(z): b for z, b in bounds.items()}
    t0 = time.time(); n_done = 0
    for b0 in range(0, len(todo), batch_size):
        batch = todo[b0:b0 + batch_size]
        wait_for_disk(cfg["disk_path"])
        shutil.rmtree(cfg["extract"], ignore_errors=True); os.makedirs(cfg["extract"])
        if cfg.get("fetch"):              # sheets come as single files (e.g. from a SharePoint folder)
            cfg["fetch"](batch, cfg["extract"])
        else:                             # sheets come out of the 7z archive
            lst = os.path.join(cfg["local"], "batch.txt")
            open(lst, "w", encoding="utf-8").write("\n".join(batch) + "\n")
            ex = subprocess.run([cfg.get("exe", "7z"), "x", cfg["first_volume"], f"-o{cfg['extract']}", f"-i@{lst}", "-y", "-bso0", "-bsp0"],
                                capture_output=True, text=True)
            if ex.returncode not in (0, 1):
                raise RuntimeError("7z extraction failed:\n" + ex.stdout[-1500:] + ex.stderr[-1500:])
        tifs, missing = [], []
        for p in batch:
            fp = os.path.join(cfg["extract"], p.replace("\\", "/"))
            (tifs if os.path.exists(fp) else missing).append((p, fp))
        results = tb.run([fp for _, fp in tifs], cfg["tiles"], 6, 14, workers=workers, bbox=bbox)
        changed = os.path.join(os.path.dirname(os.path.abspath(cfg["tiles"])), "changed_tiles.txt")
        if os.path.getsize(changed) > 0:
            for attempt in range(3):
                rc = subprocess.run(["rclone", "copy", cfg["tiles"], cfg["remote"], "--files-from", changed, "--no-traverse",
                                     "--transfers", "32"]).returncode
                if rc == 0:
                    break
                print("  upload failed, retrying ..."); time.sleep(15)
            else:
                raise RuntimeError("Upload to R2 failed 3 times. Check the R2 keys, then run this cell again.")
        rows = []
        for (p, _), r in zip(tifs, results):
            tb.merge_bounds(bounds, r["bounds"])
            c = r.get("cell", ["", ""])
            rows.append(dict(path=p, sheet=r["sheet"], status=r["status"], cell_i=c[0], cell_j=c[1],
                             cell_coverage=r.get("cell_coverage", ""), tiles=len(r["tiles"])))
        for p, _ in missing:
            rows.append(dict(path=p, sheet=os.path.basename(p), status="ERROR: not extracted", cell_i="", cell_j="", cell_coverage="", tiles=0))
        op = cfg.get("outlines_json") or os.path.join(os.path.dirname(cfg["done_csv"]), "sheet_outlines.json")
        outlines = json.load(open(op)) if os.path.exists(op) else {}
        outlines.update({r["sheet"]: r["outline"] for r in results if r.get("outline") and r["status"].startswith("ok")})
        json.dump(outlines, open(op, "w"))          # sheet name -> corners (lon, lat), for the map's sheet lookup
        new_file = not os.path.exists(cfg["done_csv"])
        with open(cfg["done_csv"], "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=DONE_FIELDS)
            if new_file:
                w.writeheader()
            w.writerows(rows)
        json.dump(bounds, open(cfg["bounds_json"], "w"))
        shutil.rmtree(cfg["extract"], ignore_errors=True)
        n_done += len(batch)
        el = time.time() - t0; left = (len(todo) - n_done) * el / n_done
        bad = [r["sheet"] + ": " + r["status"] for r in rows if not r["status"].startswith(("ok", "skipped: already", "skipped: outside"))]
        print(f"[{n_done}/{len(todo)}] batch done in total {el/60:.0f} min, about {left/60:.0f} min left" + (f"   notes: {bad}" if bad else ""))
    print("run finished")

def stitch_preview(tiles_dir, z, max_px=1600):
    """Stitch all local tiles of zoom z into one image (for a quick look)."""
    import glob
    from PIL import Image
    tiles = glob.glob(os.path.join(tiles_dir, str(z), "*", "*.png"))
    if not tiles:
        return None
    key = lambda p: (int(p.split(os.sep)[-2]), int(os.path.basename(p)[:-4]))
    xs = sorted({key(p)[0] for p in tiles}); ys = sorted({key(p)[1] for p in tiles})
    W = Image.new("RGBA", (256 * (xs[-1] - xs[0] + 1), 256 * (ys[-1] - ys[0] + 1)), (225, 225, 225, 255))
    for p in tiles:
        x, y = key(p); W.alpha_composite(Image.open(p).convert("RGBA"), ((x - xs[0]) * 256, (y - ys[0]) * 256))
    W = W.convert("RGB")
    if max(W.size) > max_px:
        W.thumbnail((max_px, max_px))
    return W

def summary(cfg):
    done = load_done(cfg["done_csv"])
    from collections import Counter
    c = Counter(r["status"].split(":")[0] for r in done.values())
    print("sheets:", len(done), dict(c))
    for r in done.values():
        if (not r["status"].startswith(("ok", "skipped: already", "skipped: outside"))) or "irregular" in r["status"] or "not clipped" in r["status"]:
            print("  ", r["sheet"], "-", r["status"], "coverage", r["cell_coverage"])
    b = json.load(open(cfg["bounds_json"])) if os.path.exists(cfg["bounds_json"]) else {}
    print("\n--- copy everything below this line to Claude ---")
    print("sheets_ok:", sum(1 for r in done.values() if r["status"].startswith("ok")), "of", len(done))
    print(f"var {cfg.get('bounds_var', 'kf2TileBounds')} = {{")
    for z in sorted(b, key=int):
        v = b[z]; print(f"    {int(z)}: {{ xMin: {v['xMin']}, xMax: {v['xMax']}, yMin: {v['yMin']}, yMax: {v['yMax']} }},")
    print("};")

def repair(cfg):
    """After an interrupted run: fix half-written tiles and put failed sheets back on the to-do list."""
    import glob
    from PIL import Image
    bad = []
    for p in glob.glob(os.path.join(cfg["tiles"], "*", "*", "*.png")):
        try:
            with Image.open(p) as im:
                im.load()
        except Exception:
            bad.append(p)
    print(len(bad), "damaged tiles found")
    for p in bad:
        rel = os.path.relpath(p, cfg["tiles"]).replace(os.sep, "/")
        os.remove(p)
        subprocess.run(["rclone", "copyto", f"{cfg['remote']}/{rel}", p], capture_output=True)   # last good copy from R2, if any
        ok = False
        if os.path.exists(p):
            try:
                with Image.open(p) as im:
                    im.load()
                ok = True
            except Exception:
                os.remove(p)
        print("  ", rel, "- restored from R2" if ok else "- removed (will be rebuilt)")
    for t in glob.glob(os.path.join(cfg["tiles"], "*", "*", "*.tmp")):
        os.remove(t)
    if os.path.exists(cfg["done_csv"]):
        rows = list(csv.DictReader(open(cfg["done_csv"], newline="", encoding="utf-8")))
        keep = [r for r in rows if not r["status"].startswith("ERROR")]
        with open(cfg["done_csv"], "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=DONE_FIELDS); w.writeheader(); w.writerows(keep)
        print(len(rows) - len(keep), "failed sheets put back on the to-do list:", [r["sheet"] for r in rows if r["status"].startswith("ERROR")])


# ---- SharePoint folder access (for packages uploaded as single files, not as one archive) ----
UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36"

class SharePointFolder:
    """Lists and downloads the files of a folder shared with an 'anyone with the link' sharing link."""
    def __init__(self, share_url):
        self.share_url = share_url
        self.login()

    def login(self):
        from urllib.parse import urlparse, parse_qs
        self.s = requests.Session(); self.s.headers["User-Agent"] = UA
        r = self.s.get(self.share_url, timeout=60)
        u = urlparse(r.url); q = parse_qs(u.query)
        if r.status_code != 200 or "id" not in q:
            raise RuntimeError(f"The sharing link did not open a folder (status {r.status_code}, landed on {r.url[:150]}). "
                               "Check that SHARE is the link of the folder itself.")
        self.root = q["id"][0]                                  # e.g. /personal/<user>/Documents/.../pc60_firstMS
        self.site = f"{u.scheme}://{u.netloc}" + "/".join(self.root.split("/")[:3])

    def _api(self, path, what):
        from urllib.parse import quote
        url = (f"{self.site}/_api/web/GetFolderByServerRelativePath(decodedurl='{quote(path.replace(chr(39), chr(39)*2))}')/{what}"
               "?$top=5000")
        for attempt in range(4):
            r = self.s.get(url, headers={"Accept": "application/json;odata=nometadata"}, timeout=60)
            if r.status_code == 200:
                return r.json()["value"]
            if r.status_code in (401, 403):
                self.login()
            time.sleep(3)
        raise RuntimeError(f"SharePoint did not list {path} (status {r.status_code}): {r.text[:300]}")

    def files(self, sub=""):
        """[(name, size, server_relative_url)] of the files directly in root/sub."""
        p = self.root + ("/" + sub.strip("/") if sub else "")
        return sorted((f["Name"], int(f["Length"]), f["ServerRelativeUrl"]) for f in self._api(p, "Files"))

    def folders(self, sub=""):
        p = self.root + ("/" + sub.strip("/") if sub else "")
        return sorted(f["Name"] for f in self._api(p, "Folders") if f["Name"] != "Forms")

    def download(self, rel_url, size, dest):
        from urllib.parse import quote
        if os.path.exists(dest) and os.path.getsize(dest) == size:
            return
        url = f"{self.site}/_layouts/15/download.aspx?SourceUrl={quote(rel_url)}"
        err = None
        for attempt in range(5):
            try:
                with self.s.get(url, stream=True, timeout=120) as r:
                    if r.status_code in (401, 403):
                        self.login(); raise IOError(f"status {r.status_code}")
                    r.raise_for_status()
                    with open(dest + ".part", "wb") as f:
                        for chunk in r.iter_content(4 * 1024**2):
                            f.write(chunk)
                if os.path.getsize(dest + ".part") != size:
                    raise IOError("size mismatch")
                os.replace(dest + ".part", dest); return
            except Exception as e:
                err = e; time.sleep(5 * (attempt + 1))
        raise RuntimeError(f"could not download {rel_url}: {err}")

    def download_many(self, items, dest_dir, threads=8):
        """items: [(name, size, rel_url)] -> dest_dir/name."""
        from concurrent.futures import ThreadPoolExecutor
        os.makedirs(dest_dir, exist_ok=True)
        with ThreadPoolExecutor(threads) as ex:
            list(ex.map(lambda it: self.download(it[2], it[1], os.path.join(dest_dir, it[0])), items))

    def download_tree(self, sub, dest_dir, threads=16):
        """Download root/sub with all its subfolders (for the small .ini files). Returns the number of files."""
        n = 0
        items = self.files(sub)
        self.download_many(items, dest_dir, threads); n += len(items)
        for d in self.folders(sub):
            n += self.download_tree(f"{sub}/{d}", os.path.join(dest_dir, d), threads)
        return n

    def find_tifs(self, sub="", skip=("raw",), depth=4):
        """All .tif files under root/sub (subfolders included, folders named in skip left out):
        [(relative_path, size, server_relative_url)]."""
        out = [((sub + "/" if sub else "") + n, size, url) for n, size, url in self.files(sub) if n.lower().endswith((".tif", ".tiff"))]
        if depth > 0:
            for d in self.folders(sub):
                if d.lower() not in skip:
                    out += self.find_tifs((sub + "/" if sub else "") + d, skip, depth - 1)
        return out


In [ ]:
%%writefile /content/work/frame.py
import numpy as np, rasterio, warnings
warnings.filterwarnings('ignore')
MAP_W, MAP_H = 7435, 4957          # expected map frame in px (18206 x 12137 m at ~2.449 m/px)
def line_profiles(path, step=4):
    with rasterio.open(path) as ds:
        a = ds.read([1,2,3], out_shape=(3, ds.height//step, ds.width//step)).astype(np.int16)
        W, H = ds.width, ds.height
    dark = (a.max(axis=0) < 110)
    return dark, W, H, step
def find_frame(path, tol=0.035):
    dark, W, H, step = line_profiles(path)
    h, w = dark.shape
    # a frame line is dark along most of its length: use the fraction of dark pixels per column / row
    col = dark.mean(axis=0); row = dark.mean(axis=1)
    def best_pair(prof, expect, n):
        exp = expect/step; best=None
        cand = [i for i in range(1,n-1) if prof[i] > 0.35 and prof[i] >= prof[i-1] and prof[i] >= prof[i+1]]
        for i in cand:
            for j in cand:
                d = j - i
                if d <= 0 or abs(d-exp) > tol*exp: continue
                s = prof[i] + prof[j] - abs(d-exp)/exp
                if best is None or s > best[0]: best = (s, i, j)
        return best, cand
    bx, cx = best_pair(col, MAP_W, w); by, cy = best_pair(row, MAP_H, h)
    return dict(W=W, H=H, x=None if not bx else (bx[1]*step, bx[2]*step), y=None if not by else (by[1]*step, by[2]*step),
                xc=[c*step for c in cx], yc=[c*step for c in cy])
if __name__ == '__main__':
    import sys
    for p in sys.argv[1:]:
        r = find_frame(p); print(p.split('/')[-1], r)


In [ ]:
%%writefile /content/work/erdely_grid_cells.py
"""Sheet number -> (column, row) of the Transylvanian First Military Survey index grid (280 sheets)."""
COLS=[67,150,232,315,396,481,564,646,731,818,903,986,1068,1151,1233,1316,1399,1481]
X0, DX = 67.0, (1481-67)/17.0
Y0, DY = 235.0-55.74, (1518-235)/23.0       # row 0 starts one row above y=235
ROWS=[(1,3)],
START={0:[(1,3)],1:[(2,2)],2:[(8,2)],3:[(16,3)],4:[(25,2)],5:[(37,3)],6:[(48,3)],7:[(59,3)],8:[(69,3)],9:[(80,3)],10:[(92,3)],11:[(104,1)],12:[(119,1)],13:[(134,1)],
       14:[(149,1)],15:[(165,0)],16:[(181,0)],17:[(198,1)],18:[(214,1)],19:[(230,0)],20:[(247,1)],21:[(263,1),(267,11)],22:[(273,12)],23:[(278,12),(279,14)]}
END={0:1,1:7,2:15,3:24,4:36,5:47,6:58,7:68,8:79,9:91,10:103,11:118,12:133,13:148,14:164,15:180,16:197,17:213,18:229,19:246,20:262,21:272,22:277,23:280}
CELL={}
for r,segs in START.items():
    for k,(n0,c0) in enumerate(segs):
        n1 = segs[k+1][0]-1 if k+1 < len(segs) else END[r]
        if r==21 and k==0: n1=266
        if r==23 and k==0: n1=278
        for n in range(n0,n1+1): CELL[n]=(c0+n-n0, r)
assert len(CELL)==280, len(CELL)


In [ ]:
%%writefile /content/work/erdely_grid_fit.json
{
 "local": "+proj=aeqd +lat_0=46.4 +lon_0=24.4 +datum=WGS84 +units=m",
 "cell": [
  18206.244864,
  12137.496576
 ],
 "similarity": [
  0.9843965411618694,
  0.2070820155729361,
  -202764.2437560081,
  130453.2048441823
 ],
 "affine": [
  [
   0.9866628172568588,
   0.2171085154869544
  ],
  [
   -0.19948692556616243,
   0.9891689380304474
  ],
  [
   -201917.25157134922,
   129552.29458551295
  ]
 ],
 "control_points": [
  [
   "083",
   3255,
   2975,
   46.7697,
   23.5895,
   "Kolozsvar, St Michael church"
  ],
  [
   "083",
   3040,
   2720,
   46.7747,
   23.5822,
   "Kolozsvar, citadel"
  ],
  [
   "053",
   1090,
   3180,
   47.1433,
   23.8747,
   "Des, centre"
  ],
  [
   "128",
   2150,
   4731,
   46.5478,
   24.5672,
   "Marosvasarhely, fortress"
  ],
  [
   "066",
   5847,
   2735,
   47.1336,
   24.4972,
   "Beszterce, church"
  ],
  [
   "170",
   4927,
   3815,
   46.068,
   23.5712,
   "Gyulafehervar, citadel"
  ],
  [
   "220",
   4733,
   4529,
   45.797,
   24.152,
   "Nagyszeben, old town"
  ],
  [
   "241",
   1059,
   3192,
   45.8452,
   24.974,
   "Fogaras, castle"
  ],
  [
   "245",
   7500,
   3750,
   46.0006,
   26.135,
   "Kezdivasarhely, west edge"
  ]
 ]
}

In [ ]:
%%writefile /content/work/erdely_cp_tool.py
"""Control-point crops for the Transylvanian First Military Survey: for a list of towns, find the sheet and the
approximate pixel (from the current grid fit), cut a crop with a labelled pixel grid and pack the crops into contact sheets."""
import json, math, os, numpy as np, rasterio, warnings
from pyproj import Transformer
from rasterio.windows import Window
from PIL import Image, ImageDraw
warnings.filterwarnings('ignore')
from erdely_grid_cells import CELL
from frame import find_frame
FIT = json.load(open(os.path.join(os.path.dirname(os.path.abspath(__file__)), 'erdely_grid_fit.json')))
CW, CH = FIT['cell']; SOL = np.array(FIT['affine']); M = SOL[:2].T; T0 = SOL[2]; MINV = np.linalg.inv(M)
TL = Transformer.from_crs("EPSG:4326", FIT['local'], always_xy=True)
INV = {v: k for k, v in CELL.items()}
TOWNS = [("Tasnad",47.477,22.584),("Szilagysomlyo",47.2306,22.8006),("Zilah",47.1806,23.0570),("Zsibo",47.2583,23.2583),("Magyarlapos",47.4603,23.8597),
 ("Naszod",47.2833,24.4069),("Radna",47.4217,24.8169),("Bethlen",47.1803,24.1792),("Szamosujvar castle",47.0300,23.9100),("Banffyhunyad",46.8667,23.0333),
 ("Gyalu castle",46.7633,23.3906),("Torda",46.5714,23.7847),("Teke",46.9167,24.5083),("Szaszregen",46.7756,24.7061),("Marosvecs castle",46.8597,24.7672),
 ("Gyergyoszentmiklos",46.7228,25.5961),("Szovata",46.5961,25.0744),("Abrudbanya",46.2742,23.0631),("Marosujvar",46.3878,23.8581),("Nagyenyed",46.3106,23.7181),
 ("Balazsfalva",46.1756,23.9156),("Dicsoszentmarton",46.3306,24.2706),("Medgyes",46.1631,24.3511),("Erzsebetvaros",46.2281,24.5750),("Segesvar",46.2194,24.7925),
 ("Szekelyudvarhely",46.3053,25.2944),("Csikszereda castle",46.3575,25.8020),("Korosbanya",46.1722,22.7178),("Zalatna",46.1100,23.2289),("Deva castle",45.8886,22.8970),
 ("Vajdahunyad castle",45.7492,22.8881),("Szaszvaros",45.8383,23.1997),("Szaszsebes",45.9569,23.5694),("Hatszeg",45.6078,22.9506),("Szerdahely",45.8850,23.7936),
 ("Nagysink",45.9167,24.8000),("Kohalom castle",46.0397,25.2111),("Sepsiszentgyorgy",45.8650,25.7870),("Brasso Black Church",45.6410,25.5880),
 ("Torcsvar castle",45.5150,25.3672),("Kovaszna",45.8494,26.1850),("Talmacs",45.6667,24.2611)]

def locate(lat, lon):
    """-> (sheet number or None, fx, fy) from the current grid fit."""
    X, Y = TL.transform(lon, lat); U, V = MINV @ (np.array([X, Y]) - T0)
    u, v = U / CW, -V / CH; c, r = math.floor(u), math.floor(v)
    return INV.get((c, r)), u - c, v - r

def crop(path, cx, cy, label, half=1200, size=640):
    with rasterio.open(path) as ds:
        x0 = int(max(0, min(ds.width - 2*half, cx - half))); y0 = int(max(0, min(ds.height - 2*half, cy - half)))
        a = ds.read([1, 2, 3], window=Window(x0, y0, 2*half, 2*half), out_shape=(3, size, size))
    im = Image.fromarray(a.transpose(1, 2, 0)); d = ImageDraw.Draw(im); k = size / (2*half)
    for v in range((x0 // 250 + 1) * 250, x0 + 2*half, 250):
        d.line([((v-x0)*k, 0), ((v-x0)*k, size)], fill=(255, 0, 0) if v % 1000 == 0 else (255, 170, 170), width=1)
        if v % 500 == 0: d.text(((v-x0)*k + 2, 2), str(v), fill=(255, 0, 0))
    for v in range((y0 // 250 + 1) * 250, y0 + 2*half, 250):
        d.line([(0, (v-y0)*k), (size, (v-y0)*k)], fill=(0, 0, 255) if v % 1000 == 0 else (170, 170, 255), width=1)
        if v % 500 == 0: d.text((2, (v-y0)*k + 2), str(v), fill=(0, 0, 255))
    d.rectangle([0, size-15, 8*len(label)+8, size], fill=(255, 255, 255)); d.text((4, size-13), label, fill=(0, 0, 0))
    return im

def plan():
    """[(town, lat, lon, sheet, fx, fy)] for towns that fall on a sheet."""
    out = []
    for name, lat, lon in TOWNS:
        s, fx, fy = locate(lat, lon)
        if s: out.append((name, lat, lon, s, fx, fy))
    return out

def make(sheet_path, out_dir, per_sheet=6, cols=3):
    """sheet_path(n) -> local path of sheet n (or None). Writes contact sheets + points.json into out_dir."""
    os.makedirs(out_dir, exist_ok=True); tiles = []; info = []
    for name, lat, lon, s, fx, fy in plan():
        p = sheet_path(s)
        if not p: info.append(dict(town=name, sheet=s, status='sheet not available')); continue
        fr = find_frame(p)
        if not (fr['x'] and fr['y']): info.append(dict(town=name, sheet=s, status='frame not found', size=[fr['W'], fr['H']])); continue
        px = fr['x'][0] + fx*(fr['x'][1]-fr['x'][0]); py = fr['y'][0] + fy*(fr['y'][1]-fr['y'][0])
        tiles.append(crop(p, px, py, f"{len(tiles)+1}. {name} / sheet {s:03d}"))
        info.append(dict(n=len(tiles), town=name, lat=lat, lon=lon, sheet=s, frame_x=fr['x'], frame_y=fr['y'], pred_px=[int(px), int(py)], status='ok'))
    size = tiles[0].size[0] if tiles else 640
    for g in range(0, len(tiles), per_sheet):
        grp = tiles[g:g+per_sheet]; rows = math.ceil(len(grp)/cols)
        sheet = Image.new('RGB', (cols*size, rows*size), 'white')
        for i, t in enumerate(grp): sheet.paste(t, ((i % cols)*size, (i//cols)*size))
        sheet.save(os.path.join(out_dir, f'cp_{g//per_sheet+1:02d}.jpg'), quality=88)
    json.dump(info, open(os.path.join(out_dir, 'points.json'), 'w'), indent=1)
    return info


## Run
`SHARE` is the sharing link of the `pc35-firstMS_Trans_Temes` folder.

In [ ]:
SHARE = "PASTE_THE_SHARING_LINK_OF_THE_FOLDER_HERE"
RAW = "1ms_Geotiff_export_Transylvania/raw"

import danfront_funcs as F, erdely_cp_tool as E, collections, shutil
SP = F.SharePointFolder(SHARE)
print("shared folder:", SP.root.split("/")[-1], "| contains:", SP.folders())
files = {n: (n, size, url) for n, size, url in SP.files(RAW)}
plan = E.plan()
need = sorted({f"{p[3]:03d}.tif" for p in plan})
have = [n for n in need if n in files]
print(len(plan), "towns on", len(need), "sheets;", len(have), "of them found in the folder,", round(sum(files[n][1] for n in have)/1e9, 1), "GB")
SP.download_many([files[n] for n in have], "/content/sheets", threads=6)
info = E.make(lambda n: f"/content/sheets/{n:03d}.tif" if os.path.exists(f"/content/sheets/{n:03d}.tif") else None, "/content/cp_out")
print(collections.Counter(i["status"] for i in info))
for i in info:
    if i["status"] != "ok": print("  ", i["town"], "sheet", i["sheet"], "-", i["status"])
shutil.make_archive("/content/erdely_control_points", "zip", "/content/cp_out")
from google.colab import files as gfiles
gfiles.download("/content/erdely_control_points.zip")